In [ ]:
!pip install gensim 
!pip install spacy
!pip install pyLDAvis
!pip install warnings
!pip install pandas 
!pip install numpy 
!pip install bertopic
!pip install SentenceTransformer 
!pip install HDBSCAN 
!pip install sklearn 

In [ ]:
!pip install Top2Vec

In [ ]:
!pip install bertopic

In [ ]:
!pip install Nmf

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
csvPath=''

NMF MODEL 

In [ ]:
# Topic Modelling NMF 
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
import spacy
import pyLDAvis
import pyLDAvis.gensim_models
from sklearn.datasets import fetch_20newsgroups
import warnings
import pandas as pd 
warnings.filterwarnings("ignore", category=DeprecationWarning)


def lemmatize(docs, allowed_postags = ["NOUN", "ADJ", "VERB", "ADV"]):

  nlp = spacy.load("en_core_web_sm", disable = ["parser", "ner"])
  lemmatized_docs = []
  for doc in docs:
    doc = nlp(doc)
    tokens = []
    for token in doc:
      if token.pos_ in allowed_postags:
        tokens.append(token.lemma_)
    lemmatized_docs.append(" ".join(tokens))
  return (lemmatized_docs)


def tokenize(docs):

  tokenized_docs = []
  for doc in docs:
    tokens = gensim.utils.simple_preprocess(doc, deacc=True)
    tokenized_docs.append(tokens)
  return (tokenized_docs)
query_df=pd.read_csv(csvPath)
query_df = query_df.iloc[:8000]#subset of dataframe
query_df['lyrics'] = query_df['lyrics'].astype(str)
query_df['lyrics'].head()
docs = list(query_df.loc[:, "lyrics"].values)


lemmatized_docs = lemmatize(docs)
tokenized_docs = tokenize(lemmatized_docs)


id2word = corpora.Dictionary(tokenized_docs)

# Prepare Document-Term Matrix
corpus = []
for doc in tokenized_docs:
    corpus.append(id2word.doc2bow(doc))


nmf_model = gensim.models.Nmf(
    corpus = corpus,     # Document-Term Matrix
    id2word = id2word,   # Map word IDs to words
    num_topics = 30,     # Number of latent topics to extract
    random_state = 100,
    passes = 100,        # N° of passes through the corpus during training
    )


nmf_model.show_topics()

BERT TOPİC 

In [ ]:
#BertTopic
import pandas as pd 
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from sklearn.datasets import fetch_20newsgroups
query_df=pd.read_csv(csvPath)

query_df = query_df.iloc[:8000]#subset of dataframe
query_df['lyrics'] = query_df['lyrics'].astype(str)
query_df['lyrics'].head()

docs = list(query_df.loc[:, "lyrics"].values)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

cluster_model = HDBSCAN(min_cluster_size = 15, 
                        metric = 'euclidean', 
                        cluster_selection_method = 'eom', 
                        prediction_data = True)

# BERTopic model
topic_model = BERTopic(embedding_model = embedding_model,
                       hdbscan_model = cluster_model)

topics, probs = topic_model.fit_transform(docs)

In [ ]:
topic_model.get_topic_freq()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(top_n_topics = 25)

In [ ]:
topic_model.visualize_documents(docs)

In [ ]:
topic_model.visualize_hierarchy()

LDA MODELLİNİN KODLARI 

In [ ]:
#LDA
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
import spacy
import pyLDAvis
import pyLDAvis.gensim_models
from sklearn.datasets import fetch_20newsgroups
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


def lemmatize(docs, allowed_postags = ["NOUN", "ADJ", "VERB", "ADV"]):

  nlp = spacy.load("en_core_web_sm", disable = ["parser", "ner"])
  lemmatized_docs = []
  for doc in docs:
    doc = nlp(doc)
    tokens = []
    for token in doc:
      if token.pos_ in allowed_postags:
        tokens.append(token.lemma_)
    lemmatized_docs.append(" ".join(tokens))
  return (lemmatized_docs)


def tokenize(docs):

  tokenized_docs = []
  for doc in docs:
    tokens = gensim.utils.simple_preprocess(doc, deacc=True)
    tokenized_docs.append(tokens)
  return (tokenized_docs)

query_df=pd.read_csv(csvPath)
query_df = query_df.iloc[:8000]#subset of dataframe
query_df['lyrics'] = query_df['lyrics'].astype(str)
query_df['lyrics'].head()
docs = list(query_df.loc[:, "lyrics"].values)

lemmatized_docs = lemmatize(docs)
tokenized_docs = tokenize(lemmatized_docs)


id2word = corpora.Dictionary(tokenized_docs)


corpus = []
for doc in tokenized_docs:
    corpus.append(id2word.doc2bow(doc))


topic_model = gensim.models.ldamodel.LdaModel(
    corpus = corpus,      # Document-Term Matrix
    id2word = id2word,    # Map word IDs to words
    num_topics = 30,      # Number of latent topics to extract
    random_state = 100,
    passes = 100,         # N° of passes through the corpus during training
    )


pyLDAvis.enable_notebook()
visualization = pyLDAvis.gensim_models.prepare(
    topic_model, 
    corpus,
    id2word, 
    mds = "mmds", 
    R = 30)

visualization

TOP2VEC modellinin kodları aşağıda 

In [ ]:
#TOP2VEC
import pandas as pd
from top2vec import Top2Vec
from sklearn.datasets import fetch_20newsgroups
query_df=pd.read_csv(csvPath)

query_df = query_df.iloc[:8000]
query_df['lyrics'] = query_df['lyrics'].astype(str)
query_df['lyrics'].head()

docs = list(query_df.loc[:, "lyrics"].values)


topic_model = Top2Vec(
  docs, 
  embedding_model = 'doc2vec', # Embedding model: See [1,2] for supported models
  min_count = 50,              # Ignore words less frequent than this value
  umap_args = None,            # Dict of custom args for UMAP
  hdbscan_args = None          # Dict of custom argd for HDBSCAN
  )


topic_words, word_scores, topic_scores, topic_nums = topic_model.search_topics(
    keywords = ["lover"], 
    num_topics = 5)

for topic in topic_nums:
    topic_model.generate_topic_wordcloud(topic)

In [ ]:
topic_model.get_num_topics()

In [ ]:
topic_words, word_scores, topic_scores, topic_nums = topic_model.search_topics(
    keywords = ["live"], 
    num_topics = 5)
for topic in topic_nums:
    topic_model.generate_topic_wordcloud(topic)

In [ ]:
topic_words, word_scores, topic_nums = topic_model.get_topics()
for topic in topic_nums:
    topic_model.generate_topic_wordcloud(topic)

In [ ]:
for words, scores,num in zip(topic_words, word_scores, topic_nums):
    print(words, scores,num)
    print(f"words:{words}, scores:{scores}, num:{num}")